# One-Way ANOVA and Post-Hoc Test

This notebook demonstrates a complete one-way ANOVA workflow for comparing the mean returns of three trading strategies, followed by Tukey's HSD post-hoc test to identify which groups differ significantly.

**Decision rule:** use α = 0.05. Run the post-hoc test only when the overall ANOVA is statistically significant.

## 1. Import libraries

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

## 2. Sample data

The values below represent daily returns (%) for three independent strategies.

In [ ]:
strategy_A = np.array([
    0.10, 0.12, 0.08, 0.15, 0.11,
    0.09, 0.13, 0.10, 0.14, 0.12
])

strategy_B = np.array([
    0.11, 0.13, 0.10, 0.12, 0.14,
    0.09, 0.12, 0.11, 0.13, 0.10
])

strategy_C = np.array([
    0.20, 0.25, 0.22, 0.27, 0.24,
    0.21, 0.26, 0.23, 0.25, 0.22
])

alpha = 0.05

## 3. Calculate sample means

In [ ]:
print("Mean Returns")
print("----------------------")
print(f"Strategy A: {np.mean(strategy_A):.4f}%")
print(f"Strategy B: {np.mean(strategy_B):.4f}%")
print(f"Strategy C: {np.mean(strategy_C):.4f}%")

## 4. One-way ANOVA

### Hypotheses

- **H₀:** μA = μB = μC
- **H₁:** At least one population mean is different

The `scipy.stats.f_oneway()` function performs the one-way ANOVA.

In [ ]:
f_stat, p_value = stats.f_oneway(
    strategy_A,
    strategy_B,
    strategy_C
)

print("One-Way ANOVA")
print("----------------------")
print(f"F-statistic : {f_stat:.4f}")
print(f"P-value     : {p_value:.6f}")

## 5. ANOVA decision

In [ ]:
if p_value < alpha:
    print("ANOVA Decision: Reject H0")
    print("There is statistically significant evidence that at least one strategy has a different mean return.")
else:
    print("ANOVA Decision: Fail to reject H0")
    print("There is insufficient statistical evidence to conclude that the strategy mean returns are different.")

## 6. Prepare data for Tukey's HSD

Tukey's HSD performs pairwise comparisons while accounting for the multiple-comparison problem.

In [ ]:
returns = np.concatenate([
    strategy_A,
    strategy_B,
    strategy_C
])

groups = (
    ["Strategy A"] * len(strategy_A)
    + ["Strategy B"] * len(strategy_B)
    + ["Strategy C"] * len(strategy_C)
)

print("Total observations:", len(returns))
print("Number of groups:", len(set(groups)))

## 7. Tukey HSD post-hoc test

The post-hoc test should be interpreted after a significant overall ANOVA.

In [ ]:
if p_value < alpha:
    tukey_result = pairwise_tukeyhsd(
        endog=returns,
        groups=groups,
        alpha=alpha
    )

    print(tukey_result)
else:
    print("Tukey HSD not required because ANOVA was not statistically significant.")

## 8. Identify which groups have significantly different means

`reject = True` means the corresponding pair has a statistically significant difference at the selected alpha level.

In [ ]:
if p_value < alpha:
    tukey_table = tukey_result.summary().data

    print("Pairwise Decisions")
    print("----------------------")

    for row in tukey_table[1:]:
        group1 = row[0]
        group2 = row[1]
        p_adj = float(row[4])
        reject = bool(row[5])

        if reject:
            print(f"{group1} vs {group2}: SIGNIFICANT difference (adjusted p-value = {p_adj:.4f})")
        else:
            print(f"{group1} vs {group2}: NO significant difference (adjusted p-value = {p_adj:.4f})")

## 9. Final interpretation

Use the following logic:

1. If ANOVA p-value ≥ 0.05: fail to reject H₀. There is insufficient evidence that the group means differ.
2. If ANOVA p-value < 0.05: reject H₀. At least one mean differs, so inspect Tukey HSD.
3. In Tukey HSD, a comparison with `reject = True` indicates a statistically significant pairwise difference.

**Finance caution:** statistical significance does not automatically imply economic superiority. Also evaluate volatility, Sharpe ratio, drawdown, transaction costs, turnover, liquidity, and out-of-sample performance.

## 10. Optional: print only significant pairs

In [ ]:
if p_value < alpha:
    table = tukey_result.summary().data
    print("Significant pairwise differences:")

    found = False
    for row in table[1:]:
        if bool(row[5]):
            print(f"- {row[0]} vs {row[1]} (adjusted p-value = {float(row[4]):.4f})")
            found = True

    if not found:
        print("No significant pairwise differences were found by Tukey HSD.")
else:
    print("No post-hoc comparisons are performed because the overall ANOVA is not significant.")